# Arm Grasp Telemetry Test
Read SCSCL telemetry for IDs 1-5 and optionally command only gripper servo 4. Real motion is disabled by default. Use one notebook kernel for the serial port.

In [ ]:
from __future__ import print_function
import os, sys, time, traceback
import ipywidgets as widgets
from IPython.display import display

current = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(current, 'config.json')):
    parent = os.path.dirname(current)
    if parent == current: raise RuntimeError('project root not found')
    current = parent
PROJECT_ROOT = current
if PROJECT_ROOT not in sys.path: sys.path.insert(0, PROJECT_ROOT)
from demo_core import load_config
from demo_core.robot_control import ArmController
from tuning_tools.diagnostic_tools import append_csv, read_scscl_status, timestamped_log_path

state = {'arm': None, 'csv': None}
arm_real = widgets.Checkbox(value=False, description='arm_real')
confirm_motion = widgets.Checkbox(value=False, description='confirm gripper motion')
scenario = widgets.Text(value='empty_close', description='scenario')
servo_id = widgets.IntText(value=4, description='servo_id')
target_angle = widgets.IntText(value=-40, description='target_angle')
speed = widgets.IntText(value=120, description='speed')
capture_seconds = widgets.FloatText(value=3.0, description='capture_s')
sample_hz = widgets.FloatText(value=6.0, description='sample_hz')
output = widgets.Output(layout={'border':'1px solid #bbb','height':'420px','overflow_y':'auto'})

def controller():
    if state['arm'] is None:
        cfg = load_config(overrides={'runtime': {'dry_run': {'arm': not bool(arm_real.value)}}})
        state['arm'] = ArmController(cfg)
        state['arm'].connect()
        state['csv'] = timestamped_log_path(PROJECT_ROOT, 'arm_telemetry', 'samples.csv')
    return state['arm']

def read_all(_=None):
    with output:
        try:
            arm = controller()
            if arm.ttl is None: raise RuntimeError('arm_real is disabled; enable it before hardware reads')
            rows = [read_scscl_status(arm.ttl, sid, arm._io_lock) for sid in range(1, 6)]
            print(rows)
        except Exception: traceback.print_exc()

def capture(label, command=False):
    arm = controller()
    if arm.ttl is None: raise RuntimeError('arm_real is disabled')
    rows = []
    interval = 1.0 / max(1.0, float(sample_hz.value))
    if command:
        if not confirm_motion.value: raise RuntimeError('confirm gripper motion first')
        if int(servo_id.value) != 4: raise RuntimeError('this notebook only permits commanded motion for gripper servo 4')
        for _ in range(5):
            row = read_scscl_status(arm.ttl, 4, arm._io_lock); row.update({'timestamp':time.time(),'elapsed_s':'','scenario':label,'phase':'baseline','target_angle':int(target_angle.value)}); rows.append(row); time.sleep(interval)
        arm.move_servo(4, int(target_angle.value), int(speed.value), label)
    started = time.time()
    while time.time() - started < float(capture_seconds.value):
        row = read_scscl_status(arm.ttl, int(servo_id.value), arm._io_lock)
        row.update({'timestamp': time.time(), 'elapsed_s': time.time()-started, 'scenario': label, 'phase':'after_command' if command else 'passive', 'target_angle': int(target_angle.value) if command else ''})
        rows.append(row)
        time.sleep(interval)
    append_csv(state['csv'], rows)
    return rows

def passive(_=None):
    with output:
        try: print('saved', len(capture(scenario.value, False)), 'samples to', state['csv'])
        except Exception: traceback.print_exc()

def command_capture(_=None):
    with output:
        try: print('saved', len(capture(scenario.value, True)), 'samples to', state['csv'])
        except Exception: traceback.print_exc()

def hold(_=None):
    with output:
        try: print('hold', controller().stop_and_hold())
        except Exception: traceback.print_exc()

def release(_=None):
    with output:
        arm = state.get('arm')
        if arm is not None and arm.ttl is not None and hasattr(arm.ttl, 'portClose'): arm.ttl.portClose()
        state['arm'] = None
        print('serial released')

buttons=[]
for label, fn, style in [('Read IDs 1-5',read_all,''),('Passive Capture',passive,'info'),('Command + Capture',command_capture,'warning'),('STOP / HOLD',hold,'danger'),('Release Serial',release,'')]:
    button=widgets.Button(description=label,button_style=style); button.on_click(fn); buttons.append(button)
display(widgets.VBox([widgets.HBox([arm_real,confirm_motion,scenario]),widgets.HBox([servo_id,target_angle,speed,capture_seconds,sample_hz]),widgets.HBox(buttons),output]))
